In [1]:
%%capture
# 1. Environment & Setup
!pip install transformers torch accelerate matplotlib

import torch
import matplotlib.pyplot as plt
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Detect device
device = 0 if torch.cuda.is_available() else -1
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_str.upper()}\n")

# ========================================== #
# 2. Chapter 1/3: Pipelines
# ========================================== #
print("===")
print("CHAPTER 1/3: PIPELINES")
print("===")

# a) Sentiment Analysis
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)
sample_text = "This Google Colab notebook for the NLP assignment is incredibly helpful and easy to follow!"
sentiment_res = sentiment_analyzer(sample_text)
print(f"Sentiment Analysis Input: '{sample_text}'")
print(f"Output: {sentiment_res}\n")

# b) Zero-shot Classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
candidate_labels = ["education", "technology", "finance"]
zero_shot_res = classifier(
    "Teachers are adopting virtual assistants to customize study plans for every student.",
    candidate_labels=candidate_labels
)
print(f"Zero-shot Input: '{zero_shot_res['sequence']}'")
print(f"Output: {list(zip(zero_shot_res['labels'], zero_shot_res['scores']))}\n")

# c) Fill-Mask
fill_masker = pipeline("fill-mask", model="bert-base-uncased", device=device)
mask_res = fill_masker("The goal of life is to find your [MASK].")
print("Fill-Mask Predictions:")
for prediction in mask_res[:3]:
    print(f"  - Token: '{prediction['token_str']}' | Score: {prediction['score']:.4f}")
print()

# ========================================== #
# 3. Chapter 1/5: Transformer Architectures
# ========================================== #
print("===")
print("CHAPTER 1/5: TRANSFORMER ARCHITECTURES COMPARISON")
print("===")
comparison_summary = """
| Architecture Type  | Key Mechanism / Feature                       | Example Models | Primary Use Cases                      |
|--------------------|-----------------------------------------------|----------------|----------------------------------------|
| Encoder-only       | Bi-directional self-attention                 | BERT, RoBERTa   | Sentence classification, NER, QA       |
| Decoder-only       | Uni-directional (causal) masked self-attention | GPT family     | Text generation, creative writing      |
| Encoder-Decoder    | Seq2Seq with cross-attention layers           | T5, BART       | Translation, summarization, generation |
"""
print(comparison_summary)

# ========================================== #
# 4. Chapter 1/8: Text Generation & Inference
# ========================================== #
print("===")
print("CHAPTER 1/8: TEXT GENERATION & INFERENCE PARAMETERS")
print("===")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device_str)

# Configuration to avoid padding warning
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

prompt = "The future of artificial intelligence in education is"
inputs = tokenizer(prompt, return_tensors="pt").to(device_str)

# Define the generation strategies
strategies = {
    "Greedy Search": {"do_sample": False, "max_new_tokens": 30},
    "Beam Search": {"num_beams": 4, "early_stopping": True, "max_new_tokens": 30},
    "Low Temperature (T=0.2)": {"temperature": 0.2, "do_sample": True, "top_k": 50, "max_new_tokens": 30},
    "High Temperature (T=1.5)": {"temperature": 1.5, "do_sample": True, "top_k": 50, "max_new_tokens": 30},
    "Nucleus Sampling (p=0.9, T=0.7)": {"top_p": 0.9, "temperature": 0.7, "do_sample": True, "max_new_tokens": 30}
}

for name, params in strategies.items():
    outputs = model.generate(**inputs, **params)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"[{name}]\n{decoded.strip()}\n")

# Plot temperature probability flattening/sharpening demonstration
logits = torch.tensor([-2.5, -0.5, 1.0, 2.5, 0.2, 4.0])
temperatures = [0.2, 0.7, 1.0, 1.8]

plt.figure(figsize=(10, 6))
x_indices = range(len(logits))

for temp in temperatures:
    scaled_logits = logits / temp
    probs = torch.softmax(scaled_logits, dim=-1).numpy()
    plt.plot(x_indices, probs, marker='o', label=f'T = {temp}')

plt.title("How Temperature Adjusts Softmax Token Probabilities")
plt.xlabel("Token index (representative representation)")
plt.ylabel("Softmax Probability")
plt.xticks(x_indices, [f"Token {i}" for i in x_indices])
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()